# M01. Weather Factors

### Imports

In [1]:
from DataImports import *

### Settings

In [ ]:
ensemble_size = 10              # Number of classifiers in the model ensemble
save_event_averages = False     # Save event averages for use in WFX calculations
num_similar_weather_games = 50  # Number of similar weather games to use for WFX calibrations

### Data

##### Plate Appearances

In [3]:
pa_dataset = pd.read_csv(os.path.join(baseball_path, "PA Dataset.csv"))

##### Open Meteo

In [4]:
open_meteo_column_list = ['game_id', 'year', 'venue_name', 'location.defaultCoordinates.latitude', 'location.defaultCoordinates.longitude', 
                          'fieldInfo.leftLine', 'fieldInfo.center', 'fieldInfo.rightLine', 'fieldInfo.leftCenter', 'fieldInfo.rightCenter', 
                          'location.elevation', 'location.azimuthAngle', 'fieldInfo.roofType', 'active', 
                          'temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'surface_pressure', 
                          'wind_speed_10m', 'wind_direction_10m', 'weather_code', 'precipitation_probability']

weather_df = pd.concat(map(pd.read_csv, glob.glob(os.path.join(baseball_path, "A06. Weather", "1. Open Meteo", "*.csv"))), ignore_index=True)[open_meteo_column_list]

##### PFX

In [5]:
shifted_game_pfx_df = pd.read_csv(os.path.join(baseball_path, "PFX.csv"))

### Merge

##### MLB Stats API with Open Meteo

In [6]:
weather_dataset = pa_dataset.merge(weather_df.drop(columns=['year']), left_on=['gamePk'], right_on=['game_id'], how='inner')

##### With PFX

In [ ]:
pfx_list = [f"{event}_pfx" for event in events_list]
weather_dataset = weather_dataset.merge(shifted_game_pfx_df[['gamePk', 'batSide'] + pfx_list], on=['gamePk', 'batSide'], how='left')

### Clean

##### Open Meteo

Calculate wind vectors

In [9]:
def calculate_vectors(row, azimuth_column, wind_column, speed_column):
    angle = row[wind_column] - row[azimuth_column]
    
    # Calculate vectors
    x_vect = round(math.sin(math.radians(angle)), 5) * row[speed_column] * -1
    y_vect = round(math.cos(math.radians(angle)), 5) * row[speed_column] * -1

    return pd.Series([x_vect, y_vect], index=['x_vect', 'y_vect'])

In [10]:
weather_dataset[['meteo_x_vect', 'meteo_y_vect']] = weather_dataset.apply(lambda row: calculate_vectors(row, 'location.azimuthAngle', 'wind_direction_10m', 'wind_speed_10m'), axis=1)

Use weather column from MLB data to adjust Open Meteo data for domes/roofs <br>
Note: Second clean stage is required because it requires data from two sources

In [11]:
mask = weather_dataset['weather'].str.contains('Roof|Dome', case=False, na=False)

In [12]:
weather_dataset.loc[mask, 'temperature'] = 70
weather_dataset.loc[mask, 'x_vect'] = 0
weather_dataset.loc[mask, 'y_vect'] = 0

In [13]:
weather_dataset.loc[mask, 'temperature_2m'] = 70
weather_dataset.loc[mask, 'meteo_x_vect'] = 0
weather_dataset.loc[mask, 'meteo_y_vect'] = 0
weather_dataset.loc[mask, 'relative_humidity_2m'] = 60
weather_dataset.loc[mask, 'dew_point_2m'] = 57

### Model

$ \hat{\text{eventsModel2}} = \hat{\text{eventsModel}} + pfx + meteo\_x\_vect + meteo\_y\_vect + temperature\_2m + relative\_humidity\_2m + dew\_point\_2m + surface\_pressure + venue\_id + b\_L$

The purpose of this model is to estimate rates of events in games based on weather and venue. This model is trained with expected rates based on the actual batted ball data. This allows for us to control for differences in inherent batted ball data across games. The model then predicts with league average rates to determine how a game with typical batted ball data would differ in various weather and venue conditions. <br>
Ideally, we would then compare these predicted rates to league average rates to determine park x weather factors, multipliers that estimate how much more or less likely given events are on the game-level than under average conditions. <br>
However, this is hard. <br>
We'll recalibrate rates later using similar weather games

##### Inputs

Meteo weather inputs

In [14]:
meteo_weather_list = ['meteo_x_vect', 'meteo_y_vect', 'temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'surface_pressure']

Parks with sufficient samples

In [15]:
venue_dummy_list = [f'venue_{id}' for id in sorted(weather_dataset['venue_id'].value_counts()[lambda x: x > 20000].index.tolist())]

Predicted rates

In [16]:
events_list_pred_batted = [f"{event}_pred_batted" for event in events_list]

Select inputs

In [17]:
wfx_inputs = events_list_pred_batted + pfx_list + meteo_weather_list + venue_dummy_list + ['b_L']

##### Sample

Create venue dummies

Note: not all venue dummies may be included in venue_dummy_list

In [18]:
weather_dataset['venue_id2'] = weather_dataset['venue_id'].copy()

In [19]:
weather_dataset = pd.get_dummies(weather_dataset, columns=['venue_id2'], prefix='venue', drop_first=False)

Set pfx to 1 if not in venue sample

Note: we may want to set this in shifted_game_pfx_df and default to a rolling value

In [20]:
weather_dataset['sample_venue'] = weather_dataset[venue_dummy_list].sum(axis=1)

In [21]:
for pfx in pfx_list:
    weather_dataset[pfx] = np.where(weather_dataset['sample_venue'] == 0, 1, weather_dataset[pfx])

Drop if missing inputs

In [22]:
weather_dataset.dropna(subset=wfx_inputs, inplace=True)

Group by game

In [23]:
weather_dataset = weather_dataset.groupby(['gamePk', 'date', 'venue_id', 'batSide'])[wfx_inputs + events_list].mean().reset_index()

Define lefty batter dummy

In [24]:
weather_dataset['b_L'] = (weather_dataset['batSide'] == "L").astype(int)

Restrict by date

In [25]:
weather_dataset = weather_dataset[(weather_dataset['date'] > 20180101) & (weather_dataset['date'] < 20230101)]

Split features and target

In [26]:
X = weather_dataset[wfx_inputs].values
y = weather_dataset[events_list].values

##### Scale

In [28]:
# Columns to scale
scale_cols = [
    'b1_pred_batted','b2_pred_batted','b3_pred_batted','bb_pred_batted',
    'fo_pred_batted','go_pred_batted','hbp_pred_batted','hr_pred_batted',
    'lo_pred_batted','po_pred_batted','so_pred_batted',
    'b1_pfx','b2_pfx','b3_pfx','bb_pfx','fo_pfx','go_pfx','hbp_pfx','hr_pfx',
    'lo_pfx','po_pfx','so_pfx',
    'meteo_x_vect','meteo_y_vect','temperature_2m','relative_humidity_2m',
    'dew_point_2m','surface_pressure'
]

# Get integer column positions
scale_idx = [wfx_inputs.index(c) for c in scale_cols]

if not hasattr(sys.modules['__main__'], '__file__'):
    # Copy X to avoid modifying original
    X_scaled = X.copy()
    
    # Initialize scaler
    scale_wfx = StandardScaler()
    
    # Fit and transform only the selected columns
    X_scaled[:, scale_idx] = scale_wfx.fit_transform(X_scaled[:, scale_idx])
    
    # Create directory
    os.makedirs(
        os.path.join(model_path, "M01. Park and Weather Factors", todaysdate),
        exist_ok=True
    )
    
    # Save scaler
    pickle.dump(
        scale_wfx,
        open(
            os.path.join(model_path, "M01. Park and Weather Factors", todaysdate, "scale_wfx.pkl"),
            'wb'
        )
    )
else:
    # Transform new data with existing scaler
    X_scaled = X.copy()
    X_scaled[:, scale_idx] = scale_wfx.transform(X_scaled[:, scale_idx])

##### Train

In [ ]:
# Define a function to train one model
def train_one_model(i, X_scaled, y, model_path, todaysdate):
    model = Sequential([
        Dense(64, input_shape=(X_scaled.shape[1],), activation='relu'),
        Dense(128, activation='relu'),
        Dense(64, activation='relu'),
        Dense(32, activation='relu'),
        Dense(y.shape[1], activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.00001),
        loss=KLDivergence(),
        metrics=[KLDivergence()]
    )

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )

    model.fit(
        X_scaled, y,
        epochs=100,
        batch_size=32,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=1
    )

    # Save model
    model_path_i = os.path.join(model_path, "M01. Park and Weather Factors", todaysdate, f'predict_wfx_{i}.keras')
    model.save(model_path_i)
    return model

# Make folder
os.makedirs(os.path.join(model_path, "M01. Park and Weather Factors", todaysdate), exist_ok=True)

# Train in parallel
ensemble_models = Parallel(n_jobs=-1, backend='loky')(
    delayed(train_one_model)(i, X_scaled, y, model_path, todaysdate)
    for i in range(ensemble_size)
)

# Wrap ensemble
class VotingEnsemble:
    def __init__(self, models):
        self.models = models

    def predict(self, X):
        predictions = np.array([model.predict(X, verbose=0) for model in self.models])
        return np.mean(predictions, axis=0)

predict_wfx = VotingEnsemble(ensemble_models)

##### Averages

Sample-wide event averages. Used to calculate WFX in A06. Weather

In [30]:
average_df = pd.DataFrame(weather_dataset[events_list_pred_batted].mean()).T
if save_event_averages == True:
    average_df.to_csv(os.path.join(baseball_path, "Event Averages.csv"), index=False)

##### Sample

Replace game predicted batted ball data with averages

In [ ]:
weather_dataset2 = weather_dataset.copy()
for event in events_list:
    weather_dataset2[f'{event}_pred_batted'] = weather_dataset2[f'{event}_pred_batted'].mean()

Split features and target

In [ ]:
X2 = weather_dataset2[wfx_inputs].values
y2 = weather_dataset2[events_list].values

##### Scale

scale_wfx defined above

In [ ]:
X2_scaled = X2.copy()
scale_idx = [wfx_inputs.index(c) for c in scale_cols]
X2_scaled[:, scale_idx] = scale_wfx.transform(X2[:, scale_idx])

##### Predict

Predict using weather inputs

In [ ]:
prediction_df2 = pd.DataFrame(predict_wfx.predict(X2_scaled), columns=events_list)
prediction_df2 = prediction_df2.add_suffix('_pred_weather')

Concatenate predictions onto weather dataset

In [ ]:
prediction_df2 = pd.concat([prediction_df2, weather_dataset2.reset_index()], axis=1)

Calculate WFX

Predicted, based on weather, over predicted, based on batted-ball data

In [ ]:
for event in events_list:
    prediction_df2[f'{event}_wfx_unadj'] = prediction_df2[f'{event}_pred_weather'] / prediction_df2[f'{event}_pred_batted']

Highlight outliers

In [ ]:
# Sort and get top/bottom 500
top_500 = prediction_df2.nlargest(500, 'hr_wfx_unadj')
bottom_500 = prediction_df2.nsmallest(500, 'hr_wfx_unadj')

# Get value counts
top_counts = top_500['venue_id'].value_counts().head(5)
bottom_counts = bottom_500['venue_id'].value_counts().head(5)

# Combine into a 5x4 DataFrame
result_df = pd.DataFrame({
    'Top Venue': top_counts.index,
    'Top Count': top_counts.values,
    'Bottom Venue': bottom_counts.index,
    'Bottom Count': bottom_counts.values
})

result_df

Average weather conditions of best and worst home run games

In [ ]:
prediction_df2.sort_values('hr_wfx_unadj', ascending=False).head(100)[['meteo_y_vect', 'temperature_2m']].mean()

In [ ]:
prediction_df2.sort_values('hr_wfx_unadj', ascending=False).tail(100)[['meteo_y_vect', 'temperature_2m']].mean()

##### Plot

In [ ]:
# Adjust the number of rows and columns
n_events = len(events_list)
n_cols = 3
n_rows = (n_events + n_cols - 1) // n_cols  # Ceiling division

# Set square plots: each subplot is 5x5 inches
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
axes = axes.flatten()

for i, event in enumerate(events_list):
    ax = axes[i]
    pred_col = f"{event}_pred_weather"
    
    if pred_col not in prediction_df2.columns:
        continue

    # Bucket the predicted values into quantiles
    prediction_df2['bucket'] = pd.qcut(prediction_df2[pred_col], q=20, duplicates='drop')

    # Compute averages
    bucket_avg = prediction_df2.groupby('bucket').agg(
        avg_pred=(pred_col, 'mean'),
        avg_actual=(event, 'mean')
    ).reset_index()

    # Plot
    ax.plot(bucket_avg['avg_pred'], label='Predicted')
    ax.plot(bucket_avg['avg_actual'], label='Actual')
    ax.set_title(f"{event.upper()} Prediction vs Actual")
    ax.set_xlabel("Quantile Bucket")
    ax.set_ylabel("Rate")
    ax.legend()
    ax.grid(True)

    # Set y-axis limits: from 1/3 of the max to the max
    y_max = max(bucket_avg['avg_pred'].max(), bucket_avg['avg_actual'].max())
    y_min = y_max / 2

    # Create 10 evenly spaced ticks from y_min to y_max
    ticks = np.linspace(y_min, y_max, 10)
    ax.set_ylim(y_min, y_max)
    ax.set_yticks(np.round(ticks, 5))  # round for cleaner labels

# Remove extra axes if any
for j in range(n_events, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [ ]:
venue_rate_df = prediction_df2.groupby('venue_id').agg(hr=('hr','mean'), hr_pred_weather=('hr_pred_weather','mean'), count=('hr','count')).reset_index()
venue_rate_df['bias'] = venue_rate_df['hr_pred_weather'] / venue_rate_df['hr']
venue_rate_df['error'] = abs(venue_rate_df['hr_pred_weather'] - venue_rate_df['hr'])
venue_rate_df[venue_rate_df['count'] > 243].sort_values('bias', ascending=False)

In [ ]:
venue_rate_df[['hr_pred_weather', 'hr', 'error']].mean()

In [ ]:
venue_rate_df[['hr_pred_weather', 'hr']].corr()

Goals:
- Minimize abs(bias)
- Increase correlation

Drop bucket column 

In [ ]:
prediction_df2.drop(columns={'bucket'}, inplace=True)

##### Calibrate

In [ ]:

prediction_df2 = prediction_df2.sort_values("date").reset_index(drop=True)

# Create output columns
for event in events_list:
    prediction_df2[f"{event}_recalibrated"] = np.nan

for (venue, side), group in prediction_df2.groupby(["venue_id", "batSide"]):
    
    group = group.sort_values("date")
    idxs = group.index
    
    # Pre-pull arrays once (important for speed)
    preds_dict = {
        event: group[f"{event}_pred_weather"].values
        for event in events_list
    }
    
    actuals_dict = {
        event: group[event].values
        for event in events_list
    }
    
    for i in range(len(group)):
        if i == 0:
            continue
        
        for event in events_list:
            
            hist_preds = preds_dict[event][:i]
            hist_actuals = actuals_dict[event][:i]
            
            if len(hist_preds) == 0:
                continue
            
            diffs = np.abs(hist_preds - preds_dict[event][i])
            
            n = min(num_similar_weather_games, len(diffs))
            nearest_idx = np.argpartition(diffs, n-1)[:n]
            
            recal_value = hist_actuals[nearest_idx].mean()
            
            prediction_df2.loc[idxs[i], f"{event}_recalibrated"] = recal_value

In [ ]:
venue_rate_df2 = prediction_df2.groupby('venue_id').agg(hr=('hr','mean'), hr_pred_weather=('hr_recalibrated','mean'), count=('hr','count')).reset_index()  # Note: using recalibrated predictions here
venue_rate_df2['bias'] = venue_rate_df2['hr_pred_weather'] / venue_rate_df2['hr']
venue_rate_df2['error'] = abs(venue_rate_df2['hr_pred_weather'] - venue_rate_df2['hr'])
venue_rate_df2[venue_rate_df2['count'] > 243].sort_values('bias', ascending=False)

In [ ]:
venue_rate_df2[['hr_pred_weather', 'hr', 'error']].mean()

##### Calculate WFX

WFX = Actual for similar games / Average Batted Ball Data 

In [ ]:
for event in events_list:
    prediction_df2[f'{event}_wfx_adj'] = prediction_df2[f'{event}_recalibrated'] / prediction_df2[f'{event}_pred_batted']

##### WFX Dataframe

Convert from long to wide

In [ ]:
l_shifted_game_wfx_df = prediction_df2[prediction_df2['batSide'] == "L"]
r_shifted_game_wfx_df = prediction_df2[prediction_df2['batSide'] == "R"]

wfx_df = pd.merge(l_shifted_game_wfx_df, r_shifted_game_wfx_df, on=['venue_id', 'gamePk', 'date'], how='left', suffixes=("_l", "_r"))

Write all game WFX to CSV

In [ ]:
wfx_df[['venue_id', 'gamePk', 'date'] + [col for col in wfx_df if "wfx" in col] + [col for col in wfx_df if "pred" in col] + [f'{event}_l' for event in events_list] + [f'{event}_r' for event in events_list]].to_csv(
    os.path.join(baseball_path, "Park and Weather Factors.csv"), index=False)

Write individual-game WFX to CSV

In [ ]:
for date in wfx_df['date'].unique():
    wfx_df[wfx_df['date'] == date][['venue_id', 'gamePk', 'date'] + [col for col in wfx_df if "wfx" in col]].to_csv(os.path.join(baseball_path, "B02. WFX", f"Park and Weather Factors {date}.csv"), index=False)